# Semantic Similarity Backup Layer — Real CodeBERT Validation

Companion notebook to the structural (AST-only) competency extraction paper.

**Purpose:** validate the semantic-similarity backup signal against real Refactory
submissions, using real `microsoft/codebert-base` embeddings instead of the
bag-of-AST-tokens stub used in the sandboxed development environment (which was
shown to have no discriminative power).

**Non-negotiable design principle carried over from the structural layer:**
semantic similarity NEVER modifies `final_score`. It is a separate, labeled,
unverified signal for manual review only.

**What this notebook does, step by step:**
1. Install dependencies and confirm GPU/CPU availability
2. Upload your project files and Refactory data
3. Build real CodeBERT embeddings (replaces the `stub` backend)
4. Build a proper exemplar registry from real reference solutions
5. Build the actual test set that matters: submissions that are test-suite-correct
   but scored 0 (or near-0) structurally — the cases this layer exists to catch
6. Compute similarity, sweep thresholds, and report honestly — including if it
   *doesn't* beat the stub baseline


## 1. Setup

In [ ]:
!pip install -q transformers torch pandas scikit-learn matplotlib

import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", "cuda" if torch.cuda.is_available() else "cpu (fine for codebert-base on short snippets)")


## 2. Upload your files

Upload whichever of these you have on hand — you don't need all of them to start:

- `competency.py`, `semantic_backup.py` (your existing implementation)
- Your batch CSV report (e.g. `topk_report.csv`) — used in Step 5 to find
  submissions that are test-suite-correct but structurally 0
- A folder/zip of real Refactory submissions (`correct/`, `wrong/` per exercise)

If you'd rather pull from Google Drive instead of re-uploading each session,
use the Drive-mount cell below instead of the upload cell.


In [ ]:
from google.colab import files
uploaded = files.upload()  # select competency.py, semantic_backup.py, topk_report.csv, etc.
print(list(uploaded.keys()))


In [ ]:
# Optional: mount Google Drive instead of re-uploading every session
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_DIR = '/content/drive/MyDrive/competency_extraction'  # adjust to your path


## 3. Real CodeBERT embeddings

This mirrors `_codebert_embed` from your `semantic_backup.py` — same interface,
now actually runnable since Colab has open network access to Hugging Face.


In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

_tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")
_model = AutoModel.from_pretrained("microsoft/codebert-base")
_model.eval()
if torch.cuda.is_available():
    _model = _model.to("cuda")

@torch.no_grad()
def codebert_embed(code: str):
    inputs = _tokenizer(code, return_tensors="pt", truncation=True, max_length=512)
    if torch.cuda.is_available():
        inputs = {k: v.to("cuda") for k, v in inputs.items()}
    out = _model(**inputs)
    return out.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()

def cosine_sim(a, b) -> float:
    import numpy as np
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return float(np.dot(a, b) / (na * nb)) if na and nb else 0.0

# Sanity check: a clearly related pair should score meaningfully higher
# than a clearly unrelated pair
related_a = "def f(lst):\n    best = lst[0]\n    for x in lst:\n        if x > best:\n            best = x\n    return best"
related_b = "def f(lst):\n    return max(lst)"
unrelated  = "def greet(name):\n    return 'Hello, ' + name"

print("related vs related:", cosine_sim(codebert_embed(related_a), codebert_embed(related_b)))
print("related vs unrelated:", cosine_sim(codebert_embed(related_a), codebert_embed(unrelated)))


## 4. Real exemplar registry

Replace the 2–3 hand-written snippets from the sandboxed version with real
reference solutions pulled from your Refactory `correct/` folders — a handful
of *diverse* correct submissions per competency, not just one canonical style.

Paste in real file paths / snippets below. This is the single biggest lever on
result quality — a thin or homogeneous registry will under- or over-rate
everything compared against it.


In [ ]:
import glob

def load_exemplars(folder_glob: str, n: int = 6):
    """Grab n real correct submissions to use as exemplars for one competency."""
    paths = sorted(glob.glob(folder_glob))[:n]
    return [open(p, encoding="utf-8").read() for p in paths]

EXEMPLARS = {
    # Example — adjust paths to your actual uploaded/mounted data
    "FindExtremum": load_exemplars("refactory/data/question_5/code/correct/*.py", n=6),
    "Accumulation": load_exemplars("refactory/data/question_2/code/correct/*.py", n=6),
    # add one entry per competency you want semantic backup for
}

EXEMPLAR_EMBEDDINGS = {
    cid: [codebert_embed(snippet) for snippet in snippets]
    for cid, snippets in EXEMPLARS.items()
}
print({cid: len(v) for cid, v in EXEMPLAR_EMBEDDINGS.items()})


## 5. Build the test set that actually matters

Not "everything" — specifically: submissions that are **test-suite-correct**
but scored **0 (or near-0) on the relevant structural competency**. These are
the genuine "structurally novel but correct" cases (e.g. a `functools.reduce`
based FindExtremum with no `ast.If`, no loop, no `max()`/`min()` call) — the
paradigm case this layer exists to catch.

Load your batch CSV report (from `batch_report.py`) and filter for exactly
this group, plus a comparison group of genuinely wrong submissions that also
score 0 structurally — you want to see whether real similarity separates
"novel-correct" from "actually-wrong", not just "similar to exemplar" in
general.


In [ ]:
import pandas as pd

report = pd.read_csv("topk_report.csv")  # or whichever batch CSV you uploaded

COMPETENCY = "FindExtremum"  # adjust to the column you're validating

novel_correct = report[
    (report["label"] == "correct") &
    (report[f"{COMPETENCY}_final"] == 0)
]
genuinely_wrong = report[
    (report["label"] == "wrong") &
    (report[f"{COMPETENCY}_final"] == 0)
]

print(f"novel-correct candidates: {len(novel_correct)}")
print(f"genuinely-wrong comparison group: {len(genuinely_wrong)}")


## 6. Compute similarity and compare the two groups

If the semantic layer is doing real work, `novel_correct` similarity scores
should be *meaningfully* higher than `genuinely_wrong` scores against the same
exemplar set. If the two distributions overlap the way the stub's did
(within ~0.02 of each other), that's a real, honest negative result — report
it rather than picking a threshold that happens to look good on this sample.


In [ ]:
def max_similarity(code: str, competency_id: str) -> float:
    emb = codebert_embed(code)
    return max(cosine_sim(emb, ex) for ex in EXEMPLAR_EMBEDDINGS[competency_id])

def score_group(df, folder_prefix, competency_id):
    scores = []
    for fname in df["file"]:
        path = f"{folder_prefix}/{fname}"
        try:
            src = open(path, encoding="utf-8").read()
        except FileNotFoundError:
            continue
        scores.append(max_similarity(src, competency_id))
    return scores

# adjust folder_prefix to wherever the actual .py files live
novel_scores = score_group(novel_correct, "refactory/data/question_5/code/correct", COMPETENCY)
wrong_scores  = score_group(genuinely_wrong, "refactory/data/question_5/code/wrong", COMPETENCY)

import numpy as np
print(f"novel-correct: mean={np.mean(novel_scores):.3f} std={np.std(novel_scores):.3f} n={len(novel_scores)}")
print(f"genuinely-wrong: mean={np.mean(wrong_scores):.3f} std={np.std(wrong_scores):.3f} n={len(wrong_scores)}")


## 7. Threshold sweep

Pick a defensible cutoff from real data rather than assuming `0.5`. This uses
ROC/AUC framing: can `max_similarity` separate "novel-correct" (positive) from
"genuinely-wrong" (negative)?


In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

y_true = [1] * len(novel_scores) + [0] * len(wrong_scores)
y_score = novel_scores + wrong_scores

auc = roc_auc_score(y_true, y_score)
fpr, tpr, thresholds = roc_curve(y_true, y_score)

print(f"AUC: {auc:.3f}  (0.5 = no better than chance / the stub's failure mode)")

plt.figure(figsize=(5,5))
plt.plot(fpr, tpr, label=f"CodeBERT (AUC={auc:.2f})")
plt.plot([0,1],[0,1],'--', color='gray', label="chance")
plt.xlabel("False positive rate (genuinely-wrong flagged)")
plt.ylabel("True positive rate (novel-correct flagged)")
plt.legend()
plt.title(f"Semantic backup separability — {COMPETENCY}")
plt.show()

# Pick threshold nearest a target true-positive rate, e.g. 0.8
import numpy as np
target_tpr = 0.8
idx = np.argmin(np.abs(tpr - target_tpr))
print(f"Suggested threshold for ~{target_tpr:.0%} recall: {thresholds[idx]:.3f} "
      f"(fpr at this point: {fpr[idx]:.3f})")


## 8. Save results

Export the raw scores and the chosen threshold so they can go straight into
the companion paper's Results section, with the same honesty standard as the
structural paper's Table 4 — report the AUC and threshold you actually got,
not the one you hoped for.


In [ ]:
import pandas as pd

pd.DataFrame({
    "file": list(novel_correct["file"]) [:len(novel_scores)] + list(genuinely_wrong["file"])[:len(wrong_scores)],
    "group": ["novel_correct"] * len(novel_scores) + ["genuinely_wrong"] * len(wrong_scores),
    "similarity": novel_scores + wrong_scores,
}).to_csv("semantic_backup_validation_results.csv", index=False)

from google.colab import files
files.download("semantic_backup_validation_results.csv")
